# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Rabby0501/flyrank-ml-internship/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

**Lane: Refresh / Content Opportunity Scoring** (a core lane, not freestyle).

I picked it because I already ran its whole pipeline in notebooks 01 and 02, so I know the data and the baseline I would have to beat (a hand-written rule scored about 24% on Precision@50 in my run, a learned model about 68%). It also has a clear user and a clear action: someone with limited time needs a ranked list of pages to look at first. My plan is to build that ranked review queue, explain each score with plain reason codes, and validate it on clients the model has never seen. This lane is provisional, so I can still change it until the end of Week 4.

In [1]:
import os, pandas as pd, numpy as np
# find the starter CSV by walking up from wherever the notebook runs (Colab, local, or work/notebooks)
while not os.path.isdir("data/raw") and os.getcwd() != "/":
    os.chdir("..")
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining"] = df["trend_direction"].str.lower().eq("down")

# Lane facts I will build on: what the "declining" label means and how big the problem is in the starter sample.
print("Lane: Refresh / Content Opportunity Scoring")
print("Label (starter): a page is 'declining' when trend_direction == 'down'")
print(f"Pages in sample: {len(df):,} | pseudonymized clients: {df['client_id'].nunique()}")
print(f"Pages labelled declining: {df['is_declining'].sum():,} ({df['is_declining'].mean():.1%})")

Lane: Refresh / Content Opportunity Scoring
Label (starter): a page is 'declining' when trend_direction == 'down'
Pages in sample: 30,000 | pseudonymized clients: 32
Pages labelled declining: 16,262 (54.2%)


## 2. The question: decision, action, cost of a wrong call

**Research question:** Using only signals that were observable before the outcome, can we rank pages so that the pages shown first are more often the ones whose search impressions are declining than a simple hand-written rule manages, including for clients the model has never seen?

- **Unit of analysis:** one page (a content item) with its 90-day and 30-day search and traffic measurements.
- **Output:** a ranked review queue of pages with a score and short reason codes (for example "stale", "dropping position").
- **Decision it improves:** which pages a content team should look at first this week.
- **Action someone takes:** a content editor or SEO reviews the top pages and decides whether to refresh, rewrite, merge or leave them alone. The queue only tells them where to look; it never changes a page by itself.
- **Cost of a wrong call:** a false alarm wastes review time (each wrongly flagged page is a review that finds nothing), and a miss leaves a genuinely declining page unnoticed for longer. A page that the model ranks high but is actually healthy could also get an unnecessary edit, which is a small risk of making a good page worse, so the output must stay decision-support with a human check.
- **Why data or ML can help at all:** the team cannot read every page, and a simple rule sees only one or two signals at a time. A model can combine several signals and rank pages, but only if it beats a fair baseline on held-out clients. If it does not, the honest answer is to keep the simple rule.

In [2]:
# What a wrong call costs when a reviewer checks 50 pages.
k = 50
base_rate = df["is_declining"].mean()
print(f"If pages were picked at random, about {base_rate*k:.0f} of {k} reviewed pages would be declining")
print(f"and about {(1-base_rate)*k:.0f} of {k} reviews would be wasted on pages that are not declining.")
print("Any ranking is only useful if it clearly beats this random level on clients it has not seen.")

If pages were picked at random, about 27 of 50 reviewed pages would be declining
and about 23 of 50 reviews would be wasted on pages that are not declining.
Any ranking is only useful if it clearly beats this random level on clients it has not seen.


## 3. Quick look at the data (2-3 real numbers)

Three numbers from the starter CSV that make this lane look worth the next 7 weeks (observed on this 30,000-page anonymized sample):

1. **Declining pages are common, so there is real work to prioritise.** About 54% of pages are labelled declining.
2. **Attention is concentrated.** The top 10% of pages by impressions carry about 70% of all impressions, and declining pages account for about half of all impressions.
3. **Clients differ a lot.** Among clients with at least 100 pages, the share of declining pages ranges from 0% to 94%. That is why validation must hold out whole clients, and why a model that looks good in-sample may not carry over.

In [3]:
# Number 1: how common is a declining page?
print(f"1) Declining share of pages: {df['is_declining'].mean():.1%}  ({df['is_declining'].sum():,} of {len(df):,})")

# Number 2: how concentrated is attention, and how much of it sits on declining pages?
imp = df["impressions_90d"].sort_values(ascending=False)
top10 = imp.head(int(len(imp) * 0.10)).sum() / imp.sum()
decl_share = df.loc[df["is_declining"], "impressions_90d"].sum() / df["impressions_90d"].sum()
print(f"2) Top 10% of pages hold {top10:.1%} of impressions; declining pages hold {decl_share:.1%} of impressions")

# Number 3: how different are clients? (only clients with at least 100 pages, to avoid tiny groups)
by_client = df.groupby("client_id")["is_declining"].agg(pages="size", declining_share="mean")
by_client = by_client[by_client["pages"] >= 100]
print(f"3) Declining share across {len(by_client)} clients with >=100 pages: "
      f"min {by_client['declining_share'].min():.0%}, median {by_client['declining_share'].median():.0%}, "
      f"max {by_client['declining_share'].max():.0%}")

1) Declining share of pages: 54.2%  (16,262 of 30,000)
2) Top 10% of pages hold 70.2% of impressions; declining pages hold 51.3% of impressions
3) Declining share across 24 clients with >=100 pages: min 0%, median 55%, max 94%


## 4. Careful words: what I can and can't claim

**What my work will be able to say:** what was *observed* in this anonymized sample, *directional* patterns between page signals and a drop in impressions, and a ranked list that is *decision-support* for a human reviewer. I will report results with the validation split I used and say when a result depends on the split.

**What it will never say:**
- That a signal *causes* a page to decline. I only see associations in one snapshot.
- That I understand or can "predict Google". I am measuring this sample of pages, not the ranking algorithm.
- That a result on the starter sample is proven for all sites. Different clients behave very differently (see number 3 above).
- That a page *should* be changed. The queue says where to look first, and a person decides.

**Leakage rule I will follow:** the label comes from the recent trend, so any column that is built from that same trend (`trend_pct`, `trend_direction`, and the 30-day impression counts they are computed from) is the answer in disguise and must stay out of the model inputs. I also keep client names, URLs and private queries out of everything I publish.

In [4]:
# Careful-words check: columns I will NOT use as model inputs, because they are the label or built from it.
leaky_or_label = ["trend_direction", "trend_pct", "impressions_last_30d", "impressions_prev_30d",
                  "clicks_last_30d", "clicks_prev_30d", "sessions_last_30d", "sessions_prev_30d"]
present = [c for c in leaky_or_label if c in df.columns]
print("Excluded from model inputs (label or built from the recent trend):")
for c in present:
    print("  -", c)

# Quick proof of why: the label largely tracks the sign of trend_pct.
agree = (df["is_declining"] == (df["trend_pct"] < 0)).mean()
print(f"\nShare of pages where 'declining' agrees with trend_pct < 0: {agree:.1%} -> trend_pct largely carries the answer, so it would leak.")

Excluded from model inputs (label or built from the recent trend):
  - trend_direction
  - trend_pct
  - impressions_last_30d
  - impressions_prev_30d
  - clicks_last_30d
  - clicks_prev_30d
  - sessions_last_30d
  - sessions_prev_30d

Share of pages where 'declining' agrees with trend_pct < 0: 88.5% -> trend_pct largely carries the answer, so it would leak.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.